## Music Recommendation
### University of Virginia
### DS 7200: Distributed Computing
### Last Updated: October 15, 2025


---

#### Instructions

In this assignment, you will build an ALS recommendation algorithm based on user listening data from Autoscrobbler.

The data consists of: 
- user data (listeners)
- item data (songs)
- interaction data (user listens, which is implicit feedback).  

Additional details are in the `README` file. Be sure to review in advance.

Develop/test code to accomplish the tasks below, clearly showing the results and documenting your work.

You will use the **DataFrame API** for all work.

**TOTAL POINTS: 10**
***

#### Preprocessing Steps in order:

1) To reduce runtime, you will sample 20% of the interaction data from `user_artist_data.txt` using seed=314 (and use this seed throughout the lab)
2) Next, remove users that listened to fewer than 10 **artists** (you may ignore aliasing in this step)
3) Lastly, split the data: 60% for training / 20% for validation / 20% for testing

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import itertools

spark = SparkSession.builder.getOrCreate() # start session 
seed = 314
K = 10

data_path = "/standard/ds7200-apt4c/large_datasets/user_artist_data.txt"

data = (spark.read.option("sep", " ").schema("user INT, artist INT, count INT").csv(data_path))


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/apps/software/standard/core/pyspark/4.2.0-py3.14/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/14 21:17:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-------------------+
|value              |
+-------------------+
|1000002 1 55       |
|1000002 1000006 33 |
|1000002 1000007 8  |
|1000002 1000009 144|
|1000002 1000010 314|
+-------------------+
only showing top 5 rows
+--------------------------------------+
|value                                 |
+--------------------------------------+
|1134999\t06Crazy Life                 |
|6821360\tPang Nakarin                 |
|10113088\tTerfel, Bartoli- Mozart: Don|
|10151459\tThe Flaming Sidebur         |
|6826647\tBodenstandig 3000            |
+--------------------------------------+
only showing top 5 rows


In [2]:
#import os
#print(os.listdir("/standard/ds7200-apt4c/large_datasets/"))
#print(os.listdir("/standard/ds7200-apt4c/large_datasets/ds5110"))

In [3]:
# parse data

artist_df = (
    spark.read
    .option("sep", "\t")
    .schema("artist INT, artist_name STRING")
    .csv("artist_data.txt")
    .filter(F.col("artist").isNotNull())
)

artist_df.show(10, truncate=False)
print("Artists loaded:", artist_df.count())

# data.show(5, truncate=False)

data.printSchema()
data.show(5)
print("bad interaction rows:",
      data.filter(F.col("user").isNull() | F.col("artist").isNull() | F.col("count").isNull()).count())
data.describe("count").show()

+--------+---------------------------------+
|artist  |artist_name                      |
+--------+---------------------------------+
|1134999 |06Crazy Life                     |
|6821360 |Pang Nakarin                     |
|10113088|Terfel, Bartoli- Mozart: Don     |
|10151459|The Flaming Sidebur              |
|6826647 |Bodenstandig 3000                |
|10186265|Jota Quest e Ivete Sangalo       |
|6828986 |Toto_XX (1977                    |
|10236364|U.S Bombs -                      |
|1135000 |artist formaly know as Mat       |
|10299728|Kassierer - Musik für beide Ohren|
+--------+---------------------------------+
only showing top 10 rows
Artists loaded: 104693
root
 |-- user: integer (nullable = true)
 |-- artist: integer (nullable = true)
 |-- count: integer (nullable = true)

+-------+-------+-----+
|   user| artist|count|
+-------+-------+-----+
|1000002|      1|   55|
|1000002|1000006|   33|
|1000002|1000007|    8|
|1000002|1000009|  144|
|1000002|1000010|  314|
+-------+-

bad interaction rows: 0


[Stage 10:===========================================>              (3 + 1) / 4]

+-------+------------------+
|summary|             count|
+-------+------------------+
|  count|          24296858|
|   mean| 15.29576248089362|
| stddev|153.91532446979554|
|    min|                 1|
|    max|            439771|
+-------+------------------+



In [4]:
# preprocess here 
# you will sample 20% of the interaction data from user_artist_data.txt using seed=314
sample_data = data.sample(withReplacement = False, fraction = 0.2, seed = seed)

# remove users that listened to fewer than 10 artists (you may ignore aliasing in this step)
user_more_10 = (sample_data.groupBy("user")
                .agg(F.countDistinct("artist").alias("n_artists"))
                .filter(F.col("n_artists") >= 10)
                .select("user"))

sample_data_filtered = sample_data.join(user_more_10, "user")

# split the data: 60% for training / 20% for validation / 20% for testing

train_df, val_df, test_df = sample_data_filtered.randomSplit([0.6, 0.2, 0.2], seed = seed)

# save the data 
train_df.cache() 
val_df.cache()
test_df.cache()

# check here 
# print("Train:", train_df.count())
# print("Validation:", val_df.count())
# print("Test:", test_df.count())

Train: 2811136


Validation: 936780


[Stage 49:====================================================> (195 + 2) / 200]

Test: 937408


In [5]:
# TASK 1 (1 PT)
# Show the first row from the training set
train_df.show(1)

+----+------+-----+
|user|artist|count|
+----+------+-----+
|7340|    82|    2|
+----+------+-----+
only showing top 1 row


In [6]:
# TASK 2 (1 PT)
# Show the first row from the test set. We will refer to this user as Test User 1.
test_df.show(1)

test_user_1 = test_df.first()["user"]
print("Test User 1:", test_user_1)

+----+------+-----+
|user|artist|count|
+----+------+-----+
|7340|  1330|    3|
+----+------+-----+
only showing top 1 row
Test User 1: 7340


In [7]:
# ALS MODEL 1: Train the ALS implicit model on the training data (since the measurements are activity and not ratings)
# Use parameters: rank=8, maxIter=5, regParam=0.02, alpha=0.01

# import from libraries
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

model1 = ALS(
    userCol="user",
    itemCol="artist",
    ratingCol="count",
    coldStartStrategy="drop", 
    implicitPrefs=True,
    rank=8, maxIter=5, regParam=0.02, alpha=0.01, 
    seed = seed
)

model = model1.fit(train_df)

In [8]:
train_df.filter(F.col("user") == test_user_1).count()

15

In [18]:
# TASK 3 (2 PTS)
# Using ALS MODEL 1, make 10 recommendations for Test User 1

K = 10

test_user_1_10 = spark.createDataFrame([(test_user_1,)], ["user"])

test_user_1_recs = (model.recommendForUserSubset(test_user_1_10, K)
         .select(F.posexplode("recommendations").alias("pos", "rec"))
         .select((F.col("pos") + 1).alias("rank"),
                 F.col("rec.artist").alias("artist"),
                 F.col("rec.rating").alias("score"))
         .orderBy("rank"))

test_user_1_recs.show(truncate=False)

+----+-------+----------+
|rank|artist |score     |
+----+-------+----------+
|1   |82     |0.2337407 |
|2   |4267   |0.16141942|
|3   |930    |0.12801574|
|4   |979    |0.12552078|
|5   |1000113|0.11632391|
|6   |976    |0.11247681|
|7   |1000123|0.11130053|
|8   |1001646|0.10170575|
|9   |1394   |0.0695943 |
|10  |1001779|0.06834846|
+----+-------+----------+



In [10]:
# TASK 4 (1 PT)
# Briefly explain Recall@K in the context of recommender systems


In [26]:
# TASK 5 (1 PT)
# Calculate Recall@K for Test User 1
K = 10

def recall_at_k(model, eval_df, k = K):
    # top k recs 

    users = eval_df.select("user").distinct()
    recommendations = (model.recommendForUserSubset(users, k)
        .select("user", F.explode("recommendations.artist").alias("artist")))

    positive_cases = (recommendations.join(eval_df, ["user", "artist"])
        .groupBy("user").agg(F.count("*").alias("hits")))


    relevant_items = eval_df.groupBy("user").agg(F.count("*").alias("relevant"))

    recall_per_user = (relevant_items.join(positive_cases, "user", "left").fillna(0).withColumn("recall", F.col("hits") / F.col("relevant")))

    recall_avg = recall_per_user.agg(F.avg("recall")).first()[0]

    return recall_avg


# for test user 1
user1_test = test_df.filter(F.col("user") == test_user_1)
print("Test User 1 Recall@10 (Model 1):", recall_at_k(model, user1_test))

[Stage 456:=================================================>   (185 + 2) / 200]

Test User 1 Recall@10 (Model 1): 0.0


In [12]:
# ALS MODEL 2: For this model, you will use hyperparameter tuning on the validation set to select the best values for
# (rank, maxIter, regParam) to maximize Recall@K. Fix alpha=0.01. Fit each candidate ALS model on the training data and evaluate its recommendations on the validation set.

grid = {"rank":     [8, 16],
        "maxIter":  [5, 10],
        "regParam": [0.01, 0.1]}

results = [] 



In [13]:
# TASK 6 (2 PTS)
# Report the best values for (rank, maxIter, regParam) along with the Recall@K from the validation set


In [14]:
# Retrain the best configuration on train + validation data.


In [15]:
# TASK 7 (1 PT)
# Calculate Recall@K on the test set and report the value.


In [16]:
# TASK 8 (1 PT)
# Calculate Recall@K for Test User 1 
# Justify the result, showing the recommendations and user activity. Did hyperparameter tuning seem to help for this user?
